In [1]:
print("hello")

hello


In [2]:
import os
print(os.getcwd())

C:\Users\loq\Desktop\fraud-detection\notebooks


In [3]:
import pandas as pd
import numpy as np

train = pd.read_csv("../donnees/processed/train.csv")
print(train.shape)
train.head()

(227845, 31)


,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Class
0,161919.0,1.946747,-0.752526,-1.355130,-0.661630,1.502822,4.024933,-1.479661,1.139880,1.406819,...,0.076197,0.297537,0.307915,0.690980,-0.350316,-0.388907,0.077641,-0.032248,7.32,0
1,124477.0,2.035149,-0.048880,-3.058693,0.247945,2.943487,3.298697,-0.002192,0.674782,0.045826,...,0.038628,0.228197,0.035542,0.707090,0.512885,-0.471198,0.002520,-0.069002,2.99,0
2,41191.0,-0.991920,0.603193,0.711976,-0.992425,-0.825838,1.956261,-2.212603,-5.037523,0.000772,...,-2.798352,0.109526,-0.436530,-0.932803,0.826684,0.913773,0.038049,0.185340,175.10,0
3,132624.0,2.285718,-1.500239,-0.747565,-1.668119,-1.394143,-0.350339,-1.427984,0.010010,-1.118447,...,-0.139670,0.077013,0.208310,-0.538236,-0.278032,-0.162068,0.018045,-0.063005,6.10,0
4,59359.0,-0.448747,-1.011440,0.115903,-3.454854,0.715771,-0.147490,0.504347,-0.113817,-0.044782,...,-0.243245,-0.173298,-0.006692,-1.362383,-0.292234,-0.144622,-0.032580,-0.064194,86.10,0


In [4]:
X_train = train.drop(columns="Class")
y_train = train["Class"]

In [5]:
def add_features(df):
    df = df.copy()
    df["Hour"] = (df["Time"] // 3600) % 24
    df["is_night"] = df["Hour"].between(0, 5).astype(int)
    df["log_amount"] = np.log1p(df["Amount"])
    return df

X_train_fe = add_features(X_train)
X_train_fe[["Time", "Hour", "is_night", "Amount", "log_amount"]].head()

,Time,Hour,is_night,Amount,log_amount
0,161919.0,20.0,0,7.32,2.118662
1,124477.0,10.0,0,2.99,1.383791
2,41191.0,11.0,0,175.10,5.171052
3,132624.0,12.0,0,6.10,1.960095
4,59359.0,16.0,0,86.10,4.467057


In [6]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer

# Colonnes à standardiser
numeric_features = ["log_amount", "Time", "Hour"]

preprocessor = ColumnTransformer(
    transformers=[
        ("scale", StandardScaler(), numeric_features),
    ],
    remainder="passthrough",   # garde V1..V28, Amount brut, is_night sans y toucher
)

In [7]:
feature_creator = FunctionTransformer(add_features)

full_preprocessing = Pipeline([
    ("features", feature_creator),
    ("preprocessing", preprocessor),
])

In [8]:
X_train_transformed = full_preprocessing.fit_transform(X_train)
print(type(X_train_transformed), X_train_transformed.shape)

<class 'numpy.ndarray'> (227845, 33)
